<a href="https://colab.research.google.com/github/inhyuk78/protein-classification-gnn/blob/main/Protein_Classification_with_Graph_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Does incorporating graph connectivity improve protein classification compared using node features alone?**

In [1]:
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_add_pool
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from collections import Counter

### **Part 0: Exploring PyTorch Geometric TUDataset**

In [3]:
dataset = TUDataset(root='data/PROTEINS', name='PROTEINS')

print(dataset)
print(f'Number of graphs (ie. individual proteins):                       {len(dataset)}')
print(f'Number of classes (ie. enzyme or non-enzyme classification):      {dataset.num_classes}')
print(f'Number of node features (ie. information explaining each node):   {dataset.num_node_features}')

Processing...


PROTEINS(1113)
Number of graphs (ie. individual proteins):                       1113
Number of classes (ie. enzyme or non-enzyme classification):      2
Number of node features (ie. information explaining each node):   3


Done!


In [4]:
# Example Protein 1113

protein = dataset[1112]

print(f'Number of nodes:        {len(protein.x)}')
print(f'Protein class:          {protein.y}')
print(f'Protein edges:          {protein.edge_index.shape}')

Number of nodes:        40
Protein class:          tensor([1])
Protein edges:          torch.Size([2, 148])


### **Part 1: Data Preparation**

In [5]:
# Train, Test split

indices = list(range(len(dataset)))

labels = []

for i in indices:
  z = dataset[i].y.item()
  labels.append(z)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    stratify=labels,
    random_state=42
)

train_dataset = dataset[train_idx]
test_dataset = dataset[test_idx]

print(f'Training Proteins: {train_dataset}')
print(f'Test Proteins: {test_dataset}')

Training Proteins: PROTEINS(890)
Test Proteins: PROTEINS(223)


In [6]:
# DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

### **Part 2: Protein Classification with a Node-Only Neural Network**

In [7]:
class NodeOnlyNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(3,16),                # Transform 3 input node features -> 16 learned features
        nn.ReLU(),
        nn.Linear(16,32),
        nn.ReLU(),
        nn.Linear(32,64),
        nn.ReLU(),
    )

    self.classifier = nn.Linear(64,2)   # Use 64 protein-level features -> 2 class score [score for class 0, score for class 1]

  def forward(self,x,batch):
    x = self.network(x)
    x = global_add_pool(x,batch)       # Sum of node representations within each protein: [N,64] -> [1,64]
    x = self.classifier(x)

    return x

model = NodeOnlyNN()
print(model)

NodeOnlyNN(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=64, bias=True)
    (5): ReLU()
  )
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


In [8]:
lr = 0.01

optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

In [9]:
# Model Training

epochs = 150

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch in train_loader:
    # 1) Forward Pass
    output = model(batch.x, batch.batch)

    # 2) Calculate Loss
    loss = loss_fn(output, batch.y)

    # 3) Backward Propagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * batch.num_graphs

  if epoch % 10 == 0:
    print(f'Epoch {epoch}: Loss = {total_loss / len(train_loader.dataset)}')

Epoch 0: Loss = 1.0285122213738689
Epoch 10: Loss = 0.5758377818579085
Epoch 20: Loss = 0.5778411565202006
Epoch 30: Loss = 0.5769534492760562
Epoch 40: Loss = 0.5753635924853636
Epoch 50: Loss = 0.591067800361119
Epoch 60: Loss = 0.5860281245092328
Epoch 70: Loss = 0.5807729115646877
Epoch 80: Loss = 0.589330523737361
Epoch 90: Loss = 0.5765616005726075
Epoch 100: Loss = 0.575700373997849
Epoch 110: Loss = 0.5828036014953356
Epoch 120: Loss = 0.5742418862460704
Epoch 130: Loss = 0.5748889640475927
Epoch 140: Loss = 0.5771589655554696


In [10]:
# Model Evaluation

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
  for batch in test_loader:
    output = model(batch.x, batch.batch)
    predictions = output.argmax(dim=1)

    all_predictions.extend(predictions.numpy())
    all_labels.extend(batch.y.numpy())

accuracy_NN = accuracy_score(all_labels, all_predictions)
f1_NN = f1_score(all_labels, all_predictions)
print(f'Accuracy of Node-only NN: {accuracy_NN:.3f}')

Accuracy of Node-only NN: 0.735


In [11]:
print("Node-only:")
print(confusion_matrix(all_labels, all_predictions))
print("Node-only predictions:", Counter(all_predictions))

Node-only:
[[100  33]
 [ 26  64]]
Node-only predictions: Counter({np.int64(0): 126, np.int64(1): 97})


### **Part 3: Protein Classification with a Graph Neural Network (Node + Edges)**

In [12]:
class GCN(torch.nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = GCNConv(3,16)       # Transform 3 input node features -> 16 learned features
    self.conv2 = GCNConv(16,32)
    self.conv3 = GCNConv(32,64)

    self.classifier = nn.Linear(64,2)

  def forward(self, x, edge_index, batch):
    x = self.conv1(x, edge_index)    # Learn from node features AND neighbouring nodes (ie. Message Passing)
    x = F.relu(x)

    x = self.conv2(x, edge_index)
    x = F.relu(x)

    x = self.conv3(x, edge_index)
    x = F.relu(x)

    x = global_add_pool(x, batch)    # Sum of node representations within each protein: [N,64] -> [1,64]
    x = self.classifier(x)

    return x

model = GCN()
print(model)

GCN(
  (conv1): GCNConv(3, 16)
  (conv2): GCNConv(16, 32)
  (conv3): GCNConv(32, 64)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


In [13]:
lr = 0.01

optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = nn.CrossEntropyLoss()

In [14]:
# Model Training

epochs = 150

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch in train_loader:
    # 1) Forward Pass
    output = model(batch.x, batch.edge_index, batch.batch)

    # 2) Calculate Loss
    loss = loss_fn(output, batch.y)

    # 3) Backward Propagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item() * batch.num_graphs

  if epoch % 10 == 0:
    print(f'Epoch {epoch}: Loss = {total_loss / len(train_loader.dataset)}')

Epoch 0: Loss = 0.6225416770142116
Epoch 10: Loss = 0.5344150653046169
Epoch 20: Loss = 0.5342024764318145
Epoch 30: Loss = 0.5326288167680248
Epoch 40: Loss = 0.5507306322622835
Epoch 50: Loss = 0.5288206273920081
Epoch 60: Loss = 0.5483046202177412
Epoch 70: Loss = 0.5420361601904536
Epoch 80: Loss = 0.5275775170058347
Epoch 90: Loss = 0.5326372587278988
Epoch 100: Loss = 0.5320696173089274
Epoch 110: Loss = 0.5350542959872256
Epoch 120: Loss = 0.5312543530142709
Epoch 130: Loss = 0.5306092818131608
Epoch 140: Loss = 0.5311406406123986


In [15]:
# Model Evaluation

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
  for batch in test_loader:
    output = model(batch.x, batch.edge_index, batch.batch)
    predictions = output.argmax(dim=1)

    all_predictions.extend(predictions.numpy())
    all_labels.extend(batch.y.numpy())

accuracy_GCN = accuracy_score(all_labels, all_predictions)
f1_GCN = f1_score(all_labels, all_predictions)
print(f'Accuracy of GNN (nodes + edges): {accuracy_GCN:.3f}')

Accuracy of GNN (nodes + edges): 0.717


In [16]:
print('GNN')
print(confusion_matrix(all_labels, all_predictions))
print("GNN predictions:", Counter(all_predictions))

GNN
[[103  30]
 [ 33  57]]
GNN predictions: Counter({np.int64(0): 136, np.int64(1): 87})


### **Part 4: Comparing Node-only vs Graph-Based Models**

In [17]:
print('Node-only NN')
print(f'Accuracy:   {accuracy_NN:.3f}')
print(f'F1 Score:   {f1_NN:.3f}')

print('\nGNN')
print(f'Accuracy:   {accuracy_GCN:.3f}')
print(f'F1 Score:   {f1_GCN:.3f}')

Node-only NN
Accuracy:   0.735
F1 Score:   0.684

GNN
Accuracy:   0.717
F1 Score:   0.644


### **Part 5: Evaluation**

| Model | Accuracy | F1 Score |
|---|---|---|
| Node-only NN | 0.735 | 0.684 |
| GCN (nodes + edges) | 0.717 | 0.644 |

*Single run, stratified 80/20 split, 223 test proteins.*

**Findings**
- In this experiment, adding graph connectivity did **not** improve classification. The node-only model performed slightly better (+1.8 points accuracy, +4.0 points F1).
- However, the gap is small. A 1.8-point accuracy difference on 223 test proteins corresponds to only about 4 proteins, and re-running the notebook shifts results by a similar amount. Thus, I can only conclude that the GCN showed no benefit here, rather than that the node-only model is better.

**Possible explanations**
- The PROTEINS node features are only 3 values per node, so summing them may already capture most of the useful signal, leaving little for message passing to add.
- With only 1,113 graphs, the extra capacity of the GCN may not translate into improved generalisation.

**Limitations**
- Only one train/test split and one random seed were used, so the variability of the results is unknown.
- No validation set or hyperparameter tuning was used.
